# 01 — Inventario e split (Giorno 1)

Runner sottile: nessuna logica qui, solo orchestrazione. Tutta la
logica vive in `sharp_har/` (inventory.py, windowing.py, splits.py).
Rif. `giorno1_inventory_splits_SPEC.md`.

## 1. Setup ambiente

In [ ]:
!pip install -q -r requirements.txt

import sys
import subprocess
from pathlib import Path

REPO_URL = "<INSERIRE_URL_REPO_GITHUB>"
REPO_DIR = Path("/content/sharp-har")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

sys.path.insert(0, str(REPO_DIR))

from sharp_har.utils import set_seed, get_git_hash, read_yaml

set_seed(42)
print("git hash:", get_git_hash(REPO_DIR))

## 2. Mount Drive + staging

I dati non entrano mai nel repo: vivono su Drive, si copiano e
si scompattano localmente. Il training legge solo da `/content`.

In [ ]:
import time
import zipfile
from google.colab import drive

drive.mount("/content/drive")

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
stage_dir.mkdir(parents=True, exist_ok=True)

t0 = time.time()
for zip_name in paths_cfg["zips"]:
    src = drive_root / zip_name
    dst = Path("/content") / zip_name
    print(f"copio {src} -> {dst}")
    subprocess.run(["cp", str(src), str(dst)], check=True)
    with zipfile.ZipFile(dst) as zf:
        zf.extractall(stage_dir)
staging_seconds = time.time() - t0
print(f"staging completato in {staging_seconds:.1f}s (dato del gate giorno 2)")

## 3. Ricognizione naming

**Ispezionare l'output prima di procedere.** Se i pattern non
corrispondono a `FILENAME_PATTERN` in `sharp_har/inventory.py`,
correggere il regex lì prima di eseguire la cella successiva.

In [ ]:
from sharp_har.inventory import list_files, print_naming_patterns

files = list_files(stage_dir)
print(f"{len(files)} file trovati in {stage_dir}")
print_naming_patterns(files, n_examples=30)

## 4. Inventario

In [ ]:
from sharp_har.inventory import build_inventory

inventory_df = build_inventory(stage_dir, out_dir=REPO_DIR / "reports")

print("copertura AR-set:", sorted(inventory_df["ar_set"].unique()))
print("distribuzione dtype:")
print(inventory_df["dtype"].value_counts())
nan_pct = 100 * inventory_df["has_nan"].mean()
print(f"file con NaN: {nan_pct:.1f}%")

## 5. Verifiche / gate del giorno 1

Assi, copertura AR-set, contingenza attività×AR-set, policy NaN
(stop se >5%), decisione classi, conteggio finestre vs attesi.

In [ ]:
from sharp_har.inventory import (
    assert_axes, assert_coverage, build_contingency_table,
    apply_nan_policy, decide_classes,
)
from sharp_har.windowing import (
    count_windows, EXPECTED_WINDOWS_TRAIN_STRIDE, EXPECTED_WINDOWS_EVAL_STRIDE,
    TRAIN_STRIDE, EVAL_STRIDE,
)

assert_axes(inventory_df)

missing = assert_coverage(inventory_df)
assert not missing, f"AR-set mancanti: {missing} — blocker, risolvere prima di congelare gli split"

contingency = build_contingency_table(inventory_df, REPO_DIR / "reports" / "contingency.csv")
print(contingency)

inventory_clean = apply_nan_policy(inventory_df, out_dir=REPO_DIR / "reports")

classes_info = decide_classes(inventory_clean)
print(classes_info)

sample_n_frame = int(inventory_clean["n_frame"].median())
print("finestre attese (stride train, mediana n_frame):",
      count_windows(sample_n_frame, stride=TRAIN_STRIDE), "atteso ~", EXPECTED_WINDOWS_TRAIN_STRIDE)
print("finestre attese (stride eval, mediana n_frame):",
      count_windows(sample_n_frame, stride=EVAL_STRIDE), "atteso ~", EXPECTED_WINDOWS_EVAL_STRIDE)

## 6. Split P2-office (rotazione primaria, C1–C4)

Test = AR-8 (ufficio). Val = 15% del train stratificato. Pin delle
celle rare. Scrive `splits/p2_office.json` solo se l'assert di
copertura per-cella passa.

In [ ]:
from sharp_har.splits import build_p2_office

p2_payload = build_p2_office(
    inventory_clean,
    out_path=REPO_DIR / "splits" / "p2_office.json",
    labels=classes_info["labels"],
)
print("train:", len(p2_payload["train"]), "val:", len(p2_payload["val"]), "test:", len(p2_payload["test"]))
print("norm:", p2_payload["norm"])

## 7. Split P1-SHARP (riproduzione, C0)

Train = bedroom AR-1a. Val = 20% di AR-1a. Test = set di
generalizzazione SHARP. Scrive `splits/p1_sharp.json`.

In [ ]:
from sharp_har.splits import build_p1_sharp

p1_payload = build_p1_sharp(
    inventory_clean,
    out_path=REPO_DIR / "splits" / "p1_sharp.json",
    c0_paper_set=classes_info["c0_paper_set"],
)
print("train:", len(p1_payload["train"]), "val:", len(p1_payload["val"]), "test:", len(p1_payload["test"]))
print("norm:", p1_payload["norm"])

## 8. Commit finale

Gli artefatti prodotti in questa run sono **congelati** una volta
committati (§0.1 — non si toccano più):

- `splits/p2_office.json`, `splits/p1_sharp.json`
- `reports/inventory.csv`, `reports/contingency.csv`,
  `reports/excluded_traces.csv`, `reports/name_to_arset.json`

```bash
cd sharp-har
git add splits/*.json reports/*.csv reports/*.json
git commit -m "giorno 1: inventario + split congelati (p2_office, p1_sharp)"
git push
```